1.Load the Customer data sheet in Python. Set churned as y. Remove churned and customer_id 	from X.

2.Make a stratified 80/20 training and test split. Check how many customers canceled in each 	split. 

3.Train these five models separately: KNN, Gaussian Naive Bayes, Decision Tree, Random 	Forest, and Logistic Regression. 

4.Use a StandardScaler pipeline for KNN and Logistic Regression. Fit the scaler using training 	data only. 

5.For each model, report test accuracy, precision, recall, F1-score, and a confusion matrix. 

6.Compare the models: Which catches the most customers who cancel? Which produces the 	fewest false alarms? Would you choose the same model for both goals? 

7.Inspect what each model learned. For example, compare K values, display a small decision 	tree, examine forest feature importance, and inspect logistic regression coefficients. 

8.Write a short recommendation explaining which model you would use to identify customers 	for a retention campaign. 


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# 1. Load the data
df = pd.read_excel("../data/customer_churn_classification.xlsx", sheet_name="Customer data")
y = df["churned"]
X = df.drop(columns=["churned", "customer_id"])
print("X:", X.shape, " y:", y.shape)

C:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\_param_validation.py:11: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.4.6)
  from scipy.sparse import csr_matrix, issparse


In [ ]:
# 2. Stratified 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print("Train canceled:", y_train.sum(), "of", len(y_train))
print("Test canceled: ", y_test.sum(), "of", len(y_test))


In [ ]:
# 3 & 4. Pick K for KNN using only the training data
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = {}
for k in [1, 3, 5, 7, 9, 11, 15, 21, 31]:
    knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    scores[k] = cross_val_score(knn, X_train, y_train, cv=cv, scoring="f1").mean()
    print("K =", k, " CV F1 =", round(scores[k], 3))
best_k = max(scores, key=scores.get)
print("Best K:", best_k)

models = {
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=best_k)),
    "Gaussian NB": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(max_depth=3, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
}

In [ ]:
# 5. Train each model and score it on the test set
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print("\n" + name)
    print("accuracy ", round(accuracy_score(y_test, pred), 2))
    print("precision", round(precision_score(y_test, pred), 2))
    print("recall   ", round(recall_score(y_test, pred), 2))
    print("f1       ", round(f1_score(y_test, pred), 2))
    print(confusion_matrix(y_test, pred))

# 6: Comparing the models
- Catches the most customers who cancel: Logistic Regression. It found 21 of the 28 customers who canceled (recall 0.75).
- Fewest false alarms: Random Forest. It wrongly flagged only 7 customers who would have stayed. KNN had 8 and Logistic Regression had 11.
- Same model for both goals? No. The best model at catching churners (Logistic Regression) is not the one with the fewest false alarms (Random Forest). Catching more churners usually means contacting more people by mistake.


In [ ]:
# 7. Look inside the models
tree = models["Decision Tree"]
print(export_text(tree, feature_names=list(X.columns)))
plt.figure(figsize=(12, 6))
plot_tree(tree, feature_names=X.columns, class_names=["stay", "churn"], filled=True, fontsize=8)
plt.tight_layout()
os.makedirs("../output", exist_ok=True)     # create the Output folder if it is not there
plt.savefig("../output/tree.png", dpi=100)   # tree picture is stored in Output/

forest = models["Random Forest"]
print(pd.Series(forest.feature_importances_, index=X.columns).sort_values(ascending=False).round(3))

lr = models["Logistic Regression"][-1]
print(pd.Series(lr.coef_[0], index=X.columns).sort_values(ascending=False).round(2))

# 8: Recommendation
I would use Logistic Regression to pick customers for the retention campaign. It caught the most customers who were going to cancel (21 of 28), which is the main aim of a retention campaign. Its false alarms (11) are a little higher than the random forest's (7), but that is a small price for finding more churners. It is also easy to explain to a business team: customers with low usage, more support tickets, late payments, a competitor offer or a high fee are the ones most at risk.
If each retention offer is expensive, for example a big discount, then Random Forest is the better choice because it wastes the fewest offers. I would not use the small decision tree or KNN, since they missed too many churners.
